# INH1
## Dependencies

In [1]:
import os
import tqdm
import shutil
import numpy as np
import scipy.io
import pandas as pd
import pickle
import h5py

import logging

In [2]:
def load_mocap_data(file_path):
    mat = scipy.io.loadmat(file_path)
    exp_name = [m for m in mat.keys() if m[:2] != '__'][0]  ## TOCHANGE
    data = np.moveaxis(mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Data'][0, 0], 2, 0)
    keypoints = [label[0].replace('coordinate', 'coord') for label in
                    mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Labels'][0, 0][0]]

    # Very important. Make sure the keypoints are always in the same order even if not saved so in the original files
    new_order = np.argsort(keypoints)
    keypoints = [keypoints[n] for n in new_order]
    data = data[:, new_order, :]

    return data, keypoints

In [3]:
# check file names
file_path = "/home/group-cvg/cvg-rose/bogna_mocap/INH1A_CLB"
files = os.listdir(file_path)
print(len(files))
files

30


['INH1A_S10_M10_MC8_CLB_18_04_2019_proc_bij_14_11_2019_C.mat',
 'INH1A_S14_M4_MC7_CLB_22_04_2019_proc_bij_15_11_2019_B.mat',
 'INH1A_S12_M2_MC6_CLB_22_04_2019_proc_bij_14_11_2019_A_NO_BACK.mat',
 'INH1A_S17_M7_MC8_CLB_22_04_2019_proc_bij_18_11_2019_B.mat',
 'INH1A_S4_M4_MC7_CLB_17_04_2019_proc_bij_7_11_2019_A.mat',
 'INH1A_S22_M2_MC6_CLB_25_04_2019_proc_bij_19_11_2019_B_No_BACK.mat',
 'INH1A_S24_M4_MC7_CLB_25_04_2019_proc_bij_20_11_2019_C.mat',
 'INH1A_S29_M9_MC8_CLB_25_04_2019_proc_bij_28_11_2019_A_Very_BAD.mat',
 'INH1A_S18_M8_MC8_CLB_22_04_2019_proc_bij_19_11_2019_A.mat',
 'INH1A_S27_M7_MC8_CLB_25_04_2019_proc_bij_20_11_2019_C_NOT GOOD.mat',
 'INH1A_S9_M9_MC8_CLB_18_04_2019_proc_bij_14_1_2019_B_BAD.mat',
 'INH1A_S19_M9_MC8_CLB_22_04_2019_proc_bij_19_11_2019_C.mat',
 'INH1A_S11_M1_MC6_CLB_22_04_2019_proc_bij_14_11_2019_B.mat',
 'INH1A_S20_M10_MC8_CLB_2_04_2019_proc_bij_19_11_2019_A.mat',
 'INH1A_S13_M3_MC6_CLB_22_04_2019_proc_bij_15_11_2019_C.mat',
 'INH1A_S8_M8_MC8_CLB_18_04_2019_pr

In [ ]:
# S29 Very bad
# S27 Not Good

files.remove('INH1A_S29_M9_MC8_CLB_25_04_2019_proc_bij_28_11_2019_A_Very_BAD.mat')
files.remove('INH1A_S27_M7_MC8_CLB_25_04_2019_proc_bij_20_11_2019_C_NOT GOOD.mat') 
files.remove('INH1A_S9_M9_MC8_CLB_18_04_2019_proc_bij_14_1_2019_B_BAD.mat')


In [5]:
len(files)

28

## Process

In [6]:
data_INH1_CLB = {}

for file_name in files:
    file = os.path.join(file_path, file_name)
    
    if os.path.isfile(file):
        data, keypoints = load_mocap_data(file)
        key_name = str(file_name.split("_")[0] + "_" + file_name.split("_")[1]) + "_CLB"
        data_INH1_CLB[key_name] ={"label":None, "keypoints":None}
        data_INH1_CLB[key_name]["label"] = file_name.split(".")[-2][-1]
        data_INH1_CLB[key_name]["keypoints"] = keypoints
        # 1. Discard last dimension
        data = data[ : , : , :3]
        
        data = data[1500: ]
        
        data_INH1_CLB[key_name]["data"] = data

In [ ]:
# Correct labels
data_INH1_CLB['INH1A_S12_CLB']["label"] = "A"
data_INH1_CLB['INH1A_S22_CLB']["label"] = "B"
data_INH1_CLB['INH1A_S28_CLB']["label"] = "B"

In [ ]:
# The correct keypoint 
keypoints_name = ['left_ankle',  'left_back',  'left_coord',  'left_hip',  'left_knee', 
                  'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']

for key, value in data_INH1_CLB.items():
    
    # Create an empty array to store
    arr = np.full((18000, 10, 3), np.nan)
    i = 0  # index of correct keypoint
    j = 0  # index of actual keypoint
    while i < 10:
        if keypoints_name[i] == value["keypoints"][i]:
            arr[:, i, :] = value["data"][:, j, :]
            i += 1 
            j += 1
        else:
            value["keypoints"].insert(i, None)
            i += 1
    
    # Remove frames with too many
    missing_per_sample = np.isnan(arr).sum(axis=(1, 2))
    indices = np.where(missing_per_sample<=20)
    value["data"] = arr[indices]
    value["data"] = arr 


In [5]:
import pickle

with open('../../data/INH1_CLB.pkl', 'wb') as file:
    pickle.dump(data_INH1_CLB, file)

In [1]:
import pickle

with open('/home/rguo_hpc/myfolder/code/mocap/data/INH1_CLB.pkl', 'rb') as file:
    data_INH1_CLB = pickle.load(file)

In [20]:
import numpy as np
missing_per_sample = np.isnan(data_INH1_CLB["INH1A_S28_CLB"]["data"]).sum(axis=(1, 2))
np.where(missing_per_sample>20)[0]

array([  903,   943,   944, ..., 15624, 15625, 15626], shape=(4188,))

In [7]:
for key, value in data_INH1_CLB.items():
    print(key, value["data"].shape, value["label"])

INH1A_S10_CLB (18000, 10, 3) C
INH1A_S14_CLB (18000, 10, 3) B
INH1A_S12_CLB (18000, 10, 3) A
INH1A_S17_CLB (18000, 10, 3) B
INH1A_S4_CLB (18000, 10, 3) A
INH1A_S22_CLB (18000, 10, 3) B
INH1A_S24_CLB (18000, 10, 3) C
INH1A_S18_CLB (18000, 10, 3) A
INH1A_S19_CLB (18000, 10, 3) C
INH1A_S11_CLB (18000, 10, 3) B
INH1A_S20_CLB (18000, 10, 3) A
INH1A_S13_CLB (18000, 10, 3) C
INH1A_S8_CLB (18000, 10, 3) C
INH1A_S1_CLB (18000, 10, 3) A
INH1A_S21_CLB (18000, 10, 3) C
INH1A_S3_CLB (18000, 10, 3) B
INH1A_S6_CLB (18000, 10, 3) B
INH1A_S25_CLB (18000, 10, 3) B
INH1A_S28_CLB (18000, 10, 3) B
INH1A_S30_CLB (18000, 10, 3) B
INH1A_S2_CLB (18000, 10, 3) C
INH1A_S7_CLB (18000, 10, 3) A
INH1A_S23_CLB (18000, 10, 3) A
INH1A_S16_CLB (18000, 10, 3) C
INH1A_S5_CLB (18000, 10, 3) C
INH1A_S15_CLB (18000, 10, 3) A
INH1A_S26_CLB (18000, 10, 3) A
